In [1]:
import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def run_des(n_students, seed=42, service_range=(1, 3)):
    """
    Simulasi Discrete Event (DES) untuk pembagian lembar jawaban.
    Mengantri FIFO, 1 server (meja pengajar), waktu pelayanan Uniform.
    """
    if seed is not None:
        random.seed(seed)
        
    env = simpy.Environment()
    desk = simpy.Resource(env, capacity=1)
    records = []

    def student_process(idx):
        arrival = env.now
        with desk.request() as req:
            yield req
            wait_time = env.now - arrival
            service_time = random.uniform(*service_range)
            yield env.timeout(service_time)
            finish_time = env.now
            
            records.append({
                'mahasiswa': idx,
                'arrival': arrival,
                'wait': wait_time,
                'service': service_time,
                'finish': finish_time
            })

    # Semua mahasiswa siap di waktu 0 (FIFO berdasarkan urutan creation)
    for i in range(1, n_students + 1):
        env.process(student_process(i))
        
    env.run()
    
    df = pd.DataFrame(records)
    total_time = df['finish'].max()
    avg_wait = df['wait'].mean()
    total_service = df['service'].sum()
    utilization = (total_service / total_time * 100) if total_time > 0 else 0
    
    return df, {
        'total_time': total_time,
        'avg_wait': avg_wait,
        'utilization': utilization,
        'service_times': df['service'].values
    }

In [ ]:
def run_verification(n=30, seed=42, service_range=(1, 3)):
    print("="*60)
    print("🔍 1.2 VERIFICATION (Build the model right?)")
    print("="*60)
    
    df, metrics = run_des(n, seed, service_range)
    
    # a. Logical Flow Check
    starts = df['arrival'] + df['wait']
    overlaps = any(starts.iloc[i] < df['finish'].iloc[i-1] for i in range(1, len(df)))
    print(f"\n✅ a. Logical Flow Check: {'LULUS' if not overlaps else 'GAGAL'}")
    print("   (Tidak ada tumpang tindih pelayanan, antrian berjalan FIFO)")
    
    # b. Event Tracing
    print("\n📋 b. Event Tracing (3 Mahasiswa Pertama):")
    trace = df[['mahasiswa', 'wait', 'service', 'finish']].head(3)
    trace['mulai'] = trace['arrival'] + trace['wait']
    print(trace[['mahasiswa', 'mulai', 'finish']].to_string(index=False))
    
    # c. Extreme Condition Test
    print("\n⚡ c. Extreme Condition Test:")
    for name, n_val, sr in [("N=1", 1, (1,3)), ("Durasi Tetap=1", 30, (1,1)), ("Durasi Tetap=3", 30, (3,3))]:
        _, m = run_des(n_val, seed=123, service_range=sr)
        expected = n_val * np.mean(sr)
        status = "SESUAI" if abs(m['total_time'] - expected) < 0.01 else "MELENYIMPANG"
        print(f"   {name:20} | Sim: {m['total_time']:.2f} | Teori: {expected:.2f} | {status}")
        
    # d. Distribution Check
    print(f"\n📊 d. Pemeriksaan Distribusi: Min={df['service'].min():.2f}, Max={df['service'].max():.2f}")
    print(f"   Rentang Asumsi: {service_range} | Status: {'SESUAI' if df['service'].min()>=service_range[0] and df['service'].max()<=service_range[1] else 'TIDAK SESUAI'}")
    
    # e. Reproducibility Check
    _, m1 = run_des(n, seed=seed, service_range=service_range)
    _, m2 = run_des(n, seed=seed, service_range=service_range)
    print(f"\n🔄 e. Reproducibility Check: {'SESUAI' if m1['total_time']==m2['total_time'] else 'TIDAK KONSISTEN'}")
    
    print("\n📌 Kesimpulan Verifikasi (1.2.3): Model telah terverifikasi. Logika sistem, implementasi event, dan konsistensi seed berjalan sesuai asumsi.")
    return df, metrics

df, metrics = run_verification()

In [ ]:
def run_validation(df, metrics, n=30, service_range=(1, 3)):
    print("="*60)
    print("✅ 1.3 VALIDATION (Build the right model?)")
    print("="*60)
    
    mean_sr = np.mean(service_range)
    theoretical_total = n * mean_sr
    
    # a. Face Validation
    print("\n🗣️ a. Face Validation: Hasil simulasi (~{:.1f} menit) masuk akal secara operasional untuk {} mahasiswa.".format(metrics['total_time'], n))
    
    # b. Perhitungan Sederhana
    diff = abs(metrics['total_time'] - theoretical_total)
    print(f"\n📐 b. Perbandingan Teoritis: Simulasi={metrics['total_time']:.2f} vs Teori≈{theoretical_total:.2f}")
    print(f"   Deviasi: {diff:.2f} menit ({'MASUK AKAL' if diff < theoretical_total*0.15 else 'PERLU REVIEW'})")
    
    # c. Behavior Validation
    print("\n📈 c. Behavior Validation (Variasi N):")
    for n_val in [10, 20, 30, 40]:
        _, m = run_des(n_val, seed=42, service_range=service_range)
        print(f"   N={n_val:2d} → Total Waktu: {m['total_time']:.2f} min (Ekspektasi: meningkat)")
        
    # d. Sensitivity Analysis
    print("\n🔬 d. Sensitivity Analysis:")
    _, m_orig = run_des(n, seed=42, service_range=(1, 3))
    _, m_mod  = run_des(n, seed=42, service_range=(2, 4))
    print(f"   Rentang (1,3) → Total: {m_orig['total_time']:.2f} min")
    print(f"   Rentang (2,4) → Total: {m_mod['total_time']:.2f} min")
    print(f"   Respons Model: {'SENSITIF & SESUAI' if m_mod['total_time'] > m_orig['total_time'] else 'TIDAK RESPONSIF'}")
    
    print("\n📌 Kesimpulan Validasi (1.3.3): Hasil simulasi realistis, perilaku model konsisten dengan sistem nyata, dan layak digunakan sebagai alat analisis.")

run_validation(df, metrics)

In [ ]:
# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['service'], bins=15, ax=axes[0], kde=True, color='skyblue')
axes[0].set_title('Distribusi Waktu Pelayanan')
axes[0].axvline(np.mean(df['service']), color='red', linestyle='--', label=f'Mean={np.mean(df["service"]):.2f}')
axes[0].legend()

sns.lineplot(x=df['mahasiswa'], y=df['wait'], ax=axes[1], marker='o', color='orange')
axes[1].set_title('Waktu Tunggu vs Urutan Mahasiswa')
axes[1].set_xlabel('Mahasiswa ke-')
axes[1].set_ylabel('Waktu Tunggu (menit)')
plt.tight_layout()
plt.show()

print("="*60)
print("🏁 1.4 KESIMPULAN AKHIR")
print("="*60)
print("Model simulasi pembagian lembar jawaban telah melalui proses:")
print("✅ Verifikasi: Model dibangun dengan benar (logika FIFO, tidak overlap, reproducible).")
print("✅ Validasi: Model merepresentasikan sistem nyata (hasil mendekati teoritis, responsif terhadap perubahan parameter).")
print("📌 Model siap digunakan untuk skenario operasional variasi N dan durasi pelayanan.")